# E-Commerce Customer Behavior - Complete Exploration & Churn Prediction

**120K transactions, 10K customers, 1K products, 80K sessions, 25K reviews -- 5 interlinked tables**

---

> **TL;DR** -- This notebook provides a comprehensive EDA of a multi-table e-commerce dataset. We explore customer segments, product pricing, transaction seasonality, session conversion funnels, and review sentiment. We perform **RFM (Recency-Frequency-Monetary) analysis** and build a **churn prediction model** (ROC-AUC reported) using features from across all 5 tables. Perfect for practicing multi-table feature engineering, customer analytics, and classification.

**Contents:**
1. [Data Loading & Overview](#1)
2. [Customer Analysis](#2)
3. [Product Analysis](#3)
4. [Transaction Patterns & Seasonality](#4)
5. [Session & Conversion Analysis](#5)
6. [Review Sentiment](#6)
7. [RFM Analysis](#7)
8. [Sample ML Task: Churn Prediction](#8)
9. [Key Insights & Next Steps](#9)

---

If you find this exploration useful, please **upvote the dataset and this notebook**!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)
matplotlib.rcParams['font.size'] = 11
plt.style.use('seaborn-v0_8-whitegrid')
import warnings
warnings.filterwarnings('ignore')

# Load data
import os
base = '/kaggle/input/ecommerce-behavior' if os.path.exists('/kaggle/input/ecommerce-behavior') else '.'

customers = pd.read_csv(f'{base}/customers.csv')
products = pd.read_csv(f'{base}/products.csv')
transactions = pd.read_csv(f'{base}/transactions.csv')
sessions = pd.read_csv(f'{base}/sessions.csv')
reviews = pd.read_csv(f'{base}/reviews.csv')

print('Dataset sizes:')
for name, df in [('Customers', customers), ('Products', products),
                  ('Transactions', transactions), ('Sessions', sessions), ('Reviews', reviews)]:
    print(f'  {name:15s}: {len(df):>8,} rows x {df.shape[1]} cols')

## 1. Data Overview

In [ ]:
print('=== Customers ===')
print(customers.dtypes)
print(f'\nMissing values: {customers.isnull().sum().sum()}')
customers.head()

In [ ]:
print('=== Products ===')
products.describe().round(2)

## 2. Customer Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Segment distribution
seg_counts = customers['segment'].value_counts()
seg_counts.plot(kind='bar', ax=axes[0, 0], color=plt.cm.Set2(range(len(seg_counts))))
axes[0, 0].set_title('Customer Segments')
axes[0, 0].set_ylabel('Count')

# Age distribution
customers['age'].hist(bins=30, ax=axes[0, 1], color='steelblue', edgecolor='white')
axes[0, 1].set_title('Age Distribution')
axes[0, 1].set_xlabel('Age')

# Country distribution
customers['country'].value_counts().head(10).plot(kind='bar', ax=axes[1, 0], color='coral')
axes[1, 0].set_title('Top 10 Countries')

# Churn rate by segment
churn_by_seg = customers.groupby('segment')['is_churned'].mean().sort_values(ascending=False)
churn_by_seg.plot(kind='bar', ax=axes[1, 1], color=['#e74c3c' if v > 0.2 else '#2ecc71' for v in churn_by_seg])
axes[1, 1].set_title('Churn Rate by Segment')
axes[1, 1].set_ylabel('Churn Rate')

plt.tight_layout()
plt.show()

In [ ]:
# Lifetime value by segment
fig, ax = plt.subplots(figsize=(10, 5))
customers.boxplot(column='lifetime_value', by='segment', ax=ax)
ax.set_title('Lifetime Value by Segment')
ax.set_ylabel('LTV ($)')
plt.suptitle('')
plt.tight_layout()
plt.show()

print('LTV Statistics by Segment:')
print(customers.groupby('segment')['lifetime_value'].describe().round(2))

## 3. Product Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Price distribution by category
cat_prices = products.groupby('category')['price'].median().sort_values(ascending=False)
cat_prices.plot(kind='barh', ax=axes[0], color='teal')
axes[0].set_title('Median Price by Category')
axes[0].set_xlabel('Price ($)')

# Rating distribution
products['avg_rating'].hist(bins=20, ax=axes[1], color='gold', edgecolor='white')
axes[1].set_title('Product Rating Distribution')
axes[1].set_xlabel('Average Rating')

# Price vs Rating scatter
axes[2].scatter(products['price'], products['avg_rating'], alpha=0.3, s=10, c='purple')
axes[2].set_title('Price vs. Rating')
axes[2].set_xlabel('Price ($)')
axes[2].set_ylabel('Avg Rating')
axes[2].set_xlim(0, 500)

plt.tight_layout()
plt.show()

## 4. Transaction Patterns & Seasonality

In [ ]:
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])
transactions['month'] = transactions['transaction_date'].dt.to_period('M')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Monthly transaction volume
monthly_vol = transactions.groupby('month').size()
monthly_vol.plot(ax=axes[0, 0], marker='o', color='steelblue')
axes[0, 0].set_title('Monthly Transaction Volume')
axes[0, 0].set_ylabel('Transactions')

# Monthly revenue
monthly_rev = transactions[transactions['status'] == 'completed'].groupby('month')['total_amount'].sum()
monthly_rev.plot(ax=axes[0, 1], marker='s', color='green')
axes[0, 1].set_title('Monthly Revenue (Completed Orders)')
axes[0, 1].set_ylabel('Revenue ($)')

# Transaction status
transactions['status'].value_counts().plot(kind='pie', ax=axes[1, 0], autopct='%1.1f%%')
axes[1, 0].set_title('Transaction Status')
axes[1, 0].set_ylabel('')

# Payment methods
transactions['payment_method'].value_counts().plot(kind='bar', ax=axes[1, 1], color=plt.cm.Pastel1(range(6)))
axes[1, 1].set_title('Payment Methods')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Day of week patterns
transactions['dow'] = transactions['transaction_date'].dt.day_name()
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_counts = transactions['dow'].value_counts().reindex(dow_order)

fig, ax = plt.subplots(figsize=(10, 4))
dow_counts.plot(kind='bar', ax=ax, color='coral')
ax.set_title('Transactions by Day of Week')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 5. Session & Conversion Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Conversion rate by device
conv_by_device = sessions.groupby('device')['converted'].mean()
conv_by_device.plot(kind='bar', ax=axes[0], color=['#3498db', '#e74c3c', '#2ecc71'])
axes[0].set_title('Conversion Rate by Device')
axes[0].set_ylabel('Conversion Rate')

# Conversion rate by channel
conv_by_channel = sessions.groupby('channel')['converted'].mean().sort_values(ascending=False)
conv_by_channel.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Conversion Rate by Channel')
axes[1].set_ylabel('Conversion Rate')

# Session duration distribution
sessions['duration_seconds'].clip(upper=3000).hist(bins=50, ax=axes[2], color='green', edgecolor='white')
axes[2].set_title('Session Duration Distribution')
axes[2].set_xlabel('Duration (seconds)')

plt.tight_layout()
plt.show()

print(f'Overall conversion rate: {sessions["converted"].mean():.2%}')
print(f'Overall bounce rate: {sessions["bounced"].mean():.2%}')

## 6. Review Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Rating distribution
reviews['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0],
    color=['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60'])
axes[0].set_title('Review Rating Distribution')
axes[0].set_xlabel('Stars')
axes[0].set_ylabel('Count')

# Verified vs unverified ratings
reviews.groupby('verified_purchase')['rating'].mean().plot(kind='bar', ax=axes[1], color=['gray', 'steelblue'])
axes[1].set_title('Average Rating: Verified vs Unverified')
axes[1].set_xticklabels(['Unverified', 'Verified'])
axes[1].set_ylabel('Avg Rating')

plt.tight_layout()
plt.show()

## 7. RFM Analysis

In [ ]:
# Compute RFM (Recency, Frequency, Monetary)
completed = transactions[transactions['status'] == 'completed'].copy()
reference_date = completed['transaction_date'].max()

rfm = completed.groupby('customer_id').agg(
    recency=('transaction_date', lambda x: (reference_date - x.max()).days),
    frequency=('transaction_id', 'count'),
    monetary=('total_amount', 'sum')
).reset_index()

# Score each dimension (1-5)
rfm['R_score'] = pd.qcut(rfm['recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(['recency', 'frequency', 'monetary']):
    rfm[col].hist(bins=30, ax=axes[i], color='steelblue', edgecolor='white')
    axes[i].set_title(f'{col.title()} Distribution')
plt.tight_layout()
plt.show()

print('RFM Summary:')
print(rfm[['recency', 'frequency', 'monetary']].describe().round(2))

## 8. Sample ML Task: Churn Prediction

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

# Merge customer features with RFM
model_df = customers.merge(rfm[['customer_id', 'recency', 'frequency', 'monetary']], 
                           on='customer_id', how='left').fillna(0)

# Session features per customer
session_feats = sessions.groupby('customer_id').agg(
    total_sessions=('session_id', 'count'),
    avg_duration=('duration_seconds', 'mean'),
    total_conversions=('converted', 'sum'),
    bounce_rate=('bounced', 'mean'),
).reset_index()
model_df = model_df.merge(session_feats, on='customer_id', how='left').fillna(0)

# Encode categoricals
le_seg = LabelEncoder()
le_country = LabelEncoder()
le_gender = LabelEncoder()
model_df['segment_enc'] = le_seg.fit_transform(model_df['segment'])
model_df['country_enc'] = le_country.fit_transform(model_df['country'])
model_df['gender_enc'] = le_gender.fit_transform(model_df['gender'])

feature_cols = ['age', 'segment_enc', 'country_enc', 'gender_enc', 'lifetime_value',
                'email_opt_in', 'has_app', 'recency', 'frequency', 'monetary',
                'total_sessions', 'avg_duration', 'total_conversions', 'bounce_rate']

X = model_df[feature_cols]
y = model_df['is_churned']

# Train
clf = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=cv, scoring='roc_auc')

print(f'Churn Prediction - 5-Fold CV ROC-AUC: {scores.mean():.3f} (+/- {scores.std():.3f})')

# Feature importance
clf.fit(X, y)
feat_imp = pd.Series(clf.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
feat_imp.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature Importance for Churn Prediction')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

<a id='9'></a>
## 9. Key Insights & Next Steps

### Key Findings

1. **Customer segments** show distinct behavioral patterns -- VIPs have 10x+ higher LTV than Budget Shoppers
2. **Strong seasonality**: Q4 transaction volume is 50-100% higher than Q1-Q2 (holiday effect)
3. **Conversion rates** vary significantly by device and traffic channel -- mobile lags desktop
4. **Churn prediction** achieves good separation using RFM + session features combined
5. **Multi-table joins** enable rich feature engineering for downstream ML tasks -- the best features come from combining tables

### Ideas for Using This Dataset

| Project | Complexity | What You Learn |
|---------|------------|----------------|
| EDA + Visualization Dashboard | Beginner | pandas, matplotlib, seaborn |
| Customer Segmentation (K-Means) | Beginner | Unsupervised learning, RFM |
| Churn Prediction (XGBoost) | Intermediate | Classification, feature engineering |
| Product Recommendations (CF) | Intermediate | Collaborative filtering, similarity |
| Market Basket Analysis (Apriori) | Intermediate | Association rules, support/confidence |
| CLV Prediction (Regression) | Advanced | Survival analysis, multi-table joins |
| Demand Forecasting (Prophet) | Advanced | Time series, seasonality modeling |

### Related Competitions
- [Instacart Market Basket Analysis](https://www.kaggle.com/competitions/instacart-market-basket-analysis) -- similar multi-table structure
- [H&M Personalized Fashion Recommendations](https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations) -- recommendation systems
- [Elo Merchant Category Recommendation](https://www.kaggle.com/competitions/elo-merchant-category-recommendation) -- customer behavior

---

**Dataset by Lorenzo Scaturchio.**

### **If you found this exploration useful, please upvote both the dataset and this notebook! It helps the community discover quality resources.**